# Lekcija 3: Pragovanje, erozija i dilatacija

Pragovanje pretvara sliku u sivim tonovima u binarnu (crno-bijelu) sliku usporedbom svakog piksela s graničnom vrijednošću, odnosno pragom. Rezultat često sadrži šum, pa ga čistimo dvjema osnovnim morfološkim operacijama: **erozijom** (smanjuje bijela područja i uklanja sitne bijele točkice) i **dilatacijom** (povećava bijela područja i popunjava male rupe).


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Izrada testne slike sa šumom

Izradit ćemo umjetnu sliku u sivim tonovima sa svijetlim oblikom na tamnoj pozadini, a zatim dodati slučajni šum kako bi neki pikseli pozadine postali svijetli, a neki pikseli objekta tamni.


In [ ]:
rng = np.random.default_rng(0)

gray = np.full((120, 120), 40, dtype=np.uint8)
cv2.circle(gray, (60, 60), 35, 220, -1)

noise = rng.normal(0, 35, gray.shape)
noisy = np.clip(gray.astype(np.int16) + noise, 0, 255).astype(np.uint8)

# sprinkle a few salt-and-pepper specks
speckle_coords = rng.integers(0, 120, size=(60, 2))
for y, x in speckle_coords:
    noisy[y, x] = 255 if noisy[y, x] < 128 else 0

plt.imshow(noisy, cmap='gray', vmin=0, vmax=255)
plt.title('Noisy grayscale image')
plt.axis('off')
plt.show()

## Pragovanje

Pragovanje je jednostavna usporedba na razini pojedinačnog piksela: svaki piksel iznad praga postaje bijel (255), a svaki piksel čija je vrijednost jednaka pragu ili manja od njega postaje crn (0). U biblioteci NumPy dovoljni su jedna logička usporedba i funkcija `np.where` za odabir izlazne vrijednosti.


In [ ]:
threshold_value = 128
binary = (noisy > threshold_value) * np.uint8(255)
plt.imshow(binary, cmap='gray', vmin=0, vmax=255)
plt.title(f'Thresholded (t={threshold_value})')
plt.axis('off')
plt.show()

### Funkcija `cv2.threshold` iz biblioteke OpenCV

OpenCV objedinjuje istu operaciju u funkciji `cv2.threshold`, što je praktično jer ona podržava i varijante koje nadilaze jednostavno binarno pragovanje &mdash; invertirani rezultat, ograničavanje vrijednosti umjesto postavljanja na nulu te automatski odabir praga (`cv2.THRESH_OTSU`) &mdash; sve kroz istu funkciju. Za obično binarno pragovanje izračunava potpuno isti rezultat kao prethodno prikazana inačica s bibliotekom NumPy.


In [ ]:
_, binary_cv2 = cv2.threshold(noisy, threshold_value, 255, cv2.THRESH_BINARY)

print('cv2.threshold matches the NumPy version exactly:', np.array_equal(binary, binary_cv2))

## Otsuova metoda: automatski odabir praga

Vrijednost `threshold_value = 128` u prethodnom primjeru odabrana je ručno. **Otsuova metoda** (<a href="../references.html#otsu-1979">Otsu, 1979</a><span class="landmark-paper">&#9733;</span>) odabire prag automatski: histogram slike promatra kao mješavinu dviju klasa (objekta i pozadine) te među svim mogućim pragovima traži onaj koji minimizira varijancu *unutar klasa* (što je ekvivalentno maksimiziranju varijance *između* dviju klasa). To je prag koji najbolje razdvaja histogram u dvije kompaktne, jasno odvojene skupine. Metoda pretpostavlja da je histogram približno bimodalan, odnosno da ima dva izražena vrha, što dobro odgovara slici sa svijetlim oblikom na tamnoj pozadini.


In [ ]:
thresh_otsu, binary_otsu = cv2.threshold(noisy, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print(f'Otsu-selected threshold: {thresh_otsu:.1f}   (we hand-picked {threshold_value} above)')

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
axes[0].hist(noisy.ravel(), bins=50, color='gray')
axes[0].axvline(thresh_otsu, color='red', linestyle='--', label=f'Otsu t={thresh_otsu:.0f}')
axes[0].set_title('Histogram', fontsize=10)
axes[0].legend(fontsize=8)
axes[1].imshow(binary, cmap='gray', vmin=0, vmax=255)
axes[1].set_title(f'Manual (t={threshold_value})', fontsize=10)
axes[2].imshow(binary_otsu, cmap='gray', vmin=0, vmax=255)
axes[2].set_title(f'Otsu (t={thresh_otsu:.0f})', fontsize=10)
for ax in axes[1:]:
    ax.axis('off')
plt.tight_layout()
plt.show()

Prag koji Otsuova metoda ovdje automatski izračuna blizu je ručno odabrane vrijednosti. Otsuova metoda dobar je početni izbor kada je potrebno odabrati prag, ali imajte na umu da neće dobro raditi s histogramima u kojima su skupine slabo razdvojene ili koji imaju više vrhova.


## Erozija

Za uklanjanje šuma tipa sol i papar te neravnih rubova iz prethodnih rezultata koriste se dvije osnovne **morfološke operacije**: erozija i dilatacija. **Erozija** pomiče mali **strukturni element** (jezgru, engl. *kernel*) po slici; piksel ostaje bijel samo ako se *cijela* jezgra nalazi unutar bijelog područja. Time se bijela područja smanjuju, a sitne bijele točkice uklanjaju.


In [ ]:
kernel = np.ones((3, 3), np.uint8)
eroded = cv2.erode(binary, kernel, iterations=1)

plt.imshow(eroded, cmap='gray', vmin=0, vmax=255)
plt.title('Eroded')
plt.axis('off')
plt.show()

## Dilatacija

**Dilatacija** radi suprotno: piksel postaje bijel ako se jezgra preklapa s *bilo kojim* dijelom bijelog područja. Time se bijela područja povećavaju, a male crne rupe i točkice popunjavaju.


In [ ]:
dilated = cv2.dilate(binary, kernel, iterations=1)

plt.imshow(dilated, cmap='gray', vmin=0, vmax=255)
plt.title('Dilated')
plt.axis('off')
plt.show()

## Otvaranje: erozija pa dilatacija

Primjena erozije, a zatim dilatacije (**otvaranje**) uklanja sitne bijele točkice, a pritom vraća veličinu glavnog područja — to je uobičajen način uklanjanja šuma iz slike dobivene pragovanjem.


In [ ]:
opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, img, title in zip(
    axes,
    [binary, eroded, dilated, opened],
    ['Thresholded', 'Eroded', 'Dilated', 'Opened\n(erode then dilate)'],
):
    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Zadaci

1. Isprobajte `cv2.MORPH_CLOSE` (dilatacija pa erozija) umjesto `MORPH_OPEN`. Po čemu se razlikuju i kakav šum crnih piksela zatvaranje uklanja, a otvaranje ne?
2. Povećajte veličinu jezgre na `(5, 5)`. Kako se rezultat mijenja u usporedbi s jezgrom veličine `(3, 3)`?
